# From-Scratch HMM for Robot Localization in a Simulated Maze (with Heatmaps)

_Generated: 2025-10-22T16:12:50.138157Z_

**Objective.** Implement Hidden Markov localization on a grid world from first principles. Given a known motion model and a noisy sensor, we estimate the posterior distribution over the robot’s location over time.

**Model.** Hidden state $X_t$ is the grid cell. Observation $O_t$ is a noisy reading (e.g., cell color or local feature). We specify:
- Initial belief $b_0(i)=p(X_0=i)$
- Transition $T_{ij}=p(X_t=j\mid X_{t-1}=i, a_{t-1})$ (depends on action)
- Emission $E_j(o)=p(O_t=o\mid X_t=j)$

**Inference.** Online filtering via the forward recursion:
$$ b_t(j) = \eta \, E_j(o_t) \sum_i T_{ij} \, b_{t-1}(i), \quad \eta^{-1}=\sum_j E_j(o_t) \sum_i T_{ij} b_{t-1}(i). $$

We also include backward messages for fixed-interval smoothing and Viterbi for most likely path.

**Output.** At each time step we render a **heat map** of the belief $b_t$ over the maze.


## 0. Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)
print("\nNVIDIA SMI (if present):")
_sh("nvidia-smi || true")

In [ ]:
!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1. Imports, Seeding, and Config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

# Visualization helpers
def show_heatmap(belief, maze, title="Belief Heatmap"):
    # belief: (H,W) with zeros for walls; will display as heatmap
    fig = plt.figure(figsize=(4,4))
    plt.imshow(belief, interpolation='nearest')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 2. Maze Definition

We define a grid with free cells and walls, plus a simple **cell-color sensor**: each free cell has a discrete color label. The observation model returns the color with probability `sensor_correct`, otherwise a random incorrect color.

In [ ]:
@dataclass
class Maze:
    grid: np.ndarray        # (H,W) 0=free, 1=wall
    colors: np.ndarray      # (H,W) integer color labels for free cells; meaningful only where grid==0

def make_simple_maze():
    # 0=free, 1=wall
    grid = np.array([
        [0,0,0,0,0,0,0],
        [0,1,1,0,1,1,0],
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0]
    ], dtype=int)
    H, W = grid.shape

    # assign colors (e.g., 0..3) to free cells deterministically for variability
    colors = np.zeros_like(grid)
    c = 0
    for i in range(H):
        for j in range(W):
            if grid[i,j]==0:
                colors[i,j] = (i + 2*j + c) % 4
    return Maze(grid=grid, colors=colors)

maze = make_simple_maze()
H, W = maze.grid.shape
N = H*W
n_colors = int(maze.colors.max() + 1)
print("Maze size:", H, "x", W, "| num states:", N, "| colors:", n_colors)

## 3. State Indexing and Masks

We flatten grid cells to indices `s in {0..N-1}` but will mask out walls.

In [ ]:
def idx(i, j, W):  # 2D -> 1D
    return i*W + j

def ij(s, W):      # 1D -> 2D
    return divmod(s, W)

# Mask for free cells
free_mask = (maze.grid == 0)          # (H,W) True for free
free_states = np.where(free_mask.reshape(-1))[0]  # flattened indices of free cells
num_free = len(free_states)
print("Free cells:", num_free, "of", N)

# Helper to render a belief vector (over N) into an (H,W) image, zeroing walls.
def belief_to_grid(b, maze):
    img = np.zeros((H,W), dtype=float)
    img[free_mask] = b[free_mask.reshape(-1)]
    return img

## 4. Motion Model (Action-Conditioned Transitions)

At each step, the agent executes an action in {N,E,S,W}. With probability `p_success` it moves to the intended neighbor if free; otherwise it **stays**. With probability `p_slip` it moves to a random **other** available neighbor; remainder stays. Transitions into walls keep the agent in place.

In [ ]:
ACTIONS = ['N','E','S','W']
delta = {'N':(-1,0), 'E':(0,1), 'S':(1,0), 'W':(0,-1)}

def neighbors(i, j, maze):
    nbrs = {}
    for a in ACTIONS:
        di, dj = delta[a]
        ni, nj = i+di, j+dj
        if 0 <= ni < H and 0 <= nj < W and maze.grid[ni,nj]==0:
            nbrs[a] = (ni, nj)
    return nbrs

def transition_matrix_for_action(action, maze, p_success=0.8, p_slip=0.15):
    # Create T of shape (N,N) with rows = prev state i, cols = next state j
    # We fill only for free states; walls remain zero and self-loop
    T = np.zeros((N,N), dtype=float)
    for s in range(N):
        i, j = ij(s, W)
        if maze.grid[i,j]==1:
            T[s,s] = 1.0
            continue
        nbrs = neighbors(i, j, maze)  # dict action->(ni,nj)
        stay_prob = 1.0
        # success move
        if action in nbrs:
            t = idx(*nbrs[action], W)
            T[s, t] += p_success
            stay_prob -= p_success
        # slips to other directions
        other_dirs = [a for a in ACTIONS if a != action and a in nbrs]
        if len(other_dirs) > 0:
            slip_each = p_slip / len(other_dirs)
            for a in other_dirs:
                t = idx(*nbrs[a], W)
                T[s, t] += slip_each
            stay_prob -= p_slip
        # remaining probability stays
        T[s, s] += max(0.0, stay_prob)
        # normalize numerical noise
        row_sum = T[s].sum()
        if row_sum > 0:
            T[s] /= row_sum
        else:
            T[s, s] = 1.0
    return T

# Precompute T for each action
T_by_action = {a: transition_matrix_for_action(a, maze) for a in ACTIONS}
print("Built transition matrices for actions:", list(T_by_action.keys()))

## 5. Observation Model (Discrete Colors)

Each free cell has a color label `c(i)`. Given true state j with color c_j, the sensor emits:
- the correct color with probability `sensor_correct`,
- a uniformly random **incorrect** color with the remaining probability.
Walls are never occupied in our HMM (probability zero).

In [ ]:
def emission_vector_for_obs(obs_color, maze, sensor_correct=0.85):
    # Returns e of shape (N,), with e[j] = p(o | X=j)
    e = np.zeros(N, dtype=float)
    for s in free_states:
        i, j = ij(s, W)
        cj = maze.colors[i,j]
        if obs_color == cj:
            e[s] = sensor_correct
        else:
            # distribute remaining probability over incorrect colors
            e[s] = (1.0 - sensor_correct) / max(1, (int(maze.colors.max()+1) - 1))
    # walls remain zero
    return e

## 6. Filtering (Forward Pass)

For a sequence of actions and observations, we propagate beliefs:
$$ \bar{b}_t = T(a_{t-1})^\top b_{t-1}, \quad b_t = \eta \, e_t \odot \bar{b}_t. $$

In [ ]:
def normalize(b, eps=1e-12):
    s = b.sum()
    return b / (s + eps)

def forward_filter(actions, observations, maze, T_by_action, b0, sensor_correct=0.85):
    beliefs = [normalize(b0.copy())]
    for t, (a, o) in enumerate(zip(actions, observations), start=1):
        T = T_by_action[a]              # (N,N)
        pred = T.T @ beliefs[-1]        # prediction step
        e = emission_vector_for_obs(o, maze, sensor_correct)  # (N,)
        post = normalize(e * pred)      # update step
        beliefs.append(post)
    return beliefs  # list length T+1

## 7. Fixed-Interval Smoothing and Viterbi (Optional)

Smoothing computes $p(X_t | o_{1:T})$ using backward messages. Viterbi yields the most likely state path.

In [ ]:
def backward_messages(actions, observations, maze, T_by_action, sensor_correct=0.85):
    T = len(actions)
    betas = [None]*(T+1)
    betas[T] = np.ones(N, dtype=float)
    for t in range(T-1, -1, -1):
        a = actions[t]
        o = observations[t]
        e = emission_vector_for_obs(o, maze, sensor_correct)
        # beta_t(i) = sum_j T_ij(a) * e(j) * beta_{t+1}(j)
        betas[t] = (T_by_action[a] @ (e * betas[t+1]))
        betas[t] = normalize(betas[t])
    return betas

def smooth(beliefs, betas):
    # elementwise multiply and renormalize
    T = len(beliefs) - 1
    smoothed = [None]*(T+1)
    for t in range(T+1):
        smoothed[t] = normalize(beliefs[t] * betas[t])
    return smoothed

def viterbi(actions, observations, maze, T_by_action, b0, sensor_correct=0.85):
    T = len(actions)
    logdelta = np.full((T+1, N), -np.inf)
    psi = np.full((T+1, N), -1, dtype=int)
    # init
    e0 = emission_vector_for_obs(observations[0], maze, sensor_correct)
    # Note: first transition uses a_0 to go from X0 to X1; we handle t=1 separately
    log_b0 = np.log(normalize(b0) + 1e-12)
    T0 = T_by_action[actions[0]]
    log_e0 = np.log(e0 + 1e-12)
    # Compute logdelta[1] = log_e1 + log(T(a0)^T b0)
    pred1 = T0.T @ np.exp(log_b0)
    logdelta[1] = np.log(pred1 + 1e-12) + log_e0

    for t in range(2, T+1):
        a_prev = actions[t-1]
        o_t = observations[t-1]
        log_e = np.log(emission_vector_for_obs(o_t, maze, sensor_correct) + 1e-12)
        Tmat = T_by_action[a_prev]
        # For each next state j: max_i [logdelta[t-1,i] + log T_ij] + log e_j
        # Work in log domain
        logT = np.log(Tmat + 1e-12)
        new_logdelta = np.full(N, -np.inf)
        new_psi = np.full(N, -1, dtype=int)
        for j in range(N):
            vals = logdelta[t-1] + logT[:, j]
            arg = int(np.argmax(vals))
            new_logdelta[j] = vals[arg] + log_e[j]
            new_psi[j] = arg
        logdelta[t] = new_logdelta
        psi[t] = new_psi

    # backtrack
    path = np.zeros(T+1, dtype=int)
    path[T] = int(np.argmax(logdelta[T]))
    for t in range(T-1, 0, -1):
        path[t] = psi[t+1, path[t+1]]
    # For t=0, set same as t=1 predecessor (not unique)
    path[0] = path[1]
    return path

## 8. Simulation: Trajectory and Observations

We simulate a plausible control sequence and sensor readings from a hidden true path. You can modify `ACTIONS_SEQ_LEN`, `sensor_correct`, and slip parameters to study behavior.

In [ ]:
def random_free_cell(maze):
    idxs = np.argwhere(maze.grid==0)
    i, j = idxs[rng.integers(0, len(idxs))]
    return int(i), int(j)

# Generate a hand-crafted action sequence (alternating corridors), or random
def generate_actions(T):
    base = ['E']*3 + ['S']*2 + ['W']*3 + ['S']*1 + ['E']*2 + ['N']*2 + ['E']*2
    if len(base) >= T:
        return base[:T]
    # pad with random if needed
    extra = rng.choice(ACTIONS, size=T-len(base)).tolist()
    return base + extra

def simulate(maze, actions, sensor_correct=0.85, p_success=0.8, p_slip=0.15):
    # Build T for each action with provided motion params
    T_by_action_sim = {a: transition_matrix_for_action(a, maze, p_success=p_success, p_slip=p_slip) for a in ACTIONS}
    # start in a random free cell
    i0, j0 = random_free_cell(maze)
    s = idx(i0, j0, W)
    true_states = [s]
    observations = []
    for a in actions:
        # sample next state
        probs = T_by_action_sim[a][s]
        s = int(rng.choice(np.arange(N), p=probs))
        true_states.append(s)
        # sample observation color
        ii, jj = ij(s, W)
        true_color = maze.colors[ii, jj]
        if rng.random() < sensor_correct:
            obs = int(true_color)
        else:
            # choose an incorrect color uniformly
            choices = [c for c in range(int(maze.colors.max()+1)) if c != true_color]
            obs = int(rng.choice(choices))
        observations.append(obs)
    return true_states, observations, T_by_action_sim

# Parameters
T_len = 16
sensor_correct = 0.85
p_success = 0.8
p_slip = 0.15

actions = generate_actions(T_len)
true_states, observations, T_by_action_sim = simulate(maze, actions, sensor_correct, p_success, p_slip)

print("Actions:", actions)
print("Observations (colors):", observations)
print("True path length:", len(true_states))

## 9. Run Localization (Filtering)

We assume **known motion** and **sensor** models. Start with a uniform prior over free cells.

In [ ]:
# Initial belief: uniform over free cells
b0 = np.zeros(N, dtype=float)
b0[free_states] = 1.0 / len(free_states)

# Use the same T_by_action as in simulation (known model)
beliefs = forward_filter(actions, observations, maze, T_by_action_sim, b0, sensor_correct=sensor_correct)
print("Computed beliefs for T+1 =", len(beliefs))

## 10. Heatmaps Over Time

We render the belief grid at each time step. Note that walls are always zero.

In [ ]:
for t, b in enumerate(beliefs):
    grid_b = belief_to_grid(b, maze)
    show_heatmap(grid_b, maze, title=f"Belief Heatmap — t={t}")

## 11. Smoothing and Viterbi (Qualitative Comparison)

Smoothing can sharpen beliefs by using future evidence. Viterbi shows a single most likely path.

In [ ]:
betas = backward_messages(actions, observations, maze, T_by_action_sim, sensor_correct=sensor_correct)
smoothed = smooth(beliefs, betas)
path = viterbi(actions, observations, maze, T_by_action_sim, b0, sensor_correct=sensor_correct)

# Show terminal beliefs vs smoothed
show_heatmap(belief_to_grid(beliefs[-1], maze), maze, title="Filtering Belief at T")
show_heatmap(belief_to_grid(smoothed[-1], maze), maze, title="Smoothed Belief at T")

# Project Viterbi path to (i,j) for display
vj = [ij(s, W) for s in path]
print("Viterbi (i,j) positions:")
print(vj)

### 11.1 True vs Viterbi (textual indices)

In [ ]:
true_ij = [ij(s, W) for s in true_states]
print("True (i,j):", true_ij)
print("Viterbi (i,j):", vj)

## 12. Save Artifacts & Download

We persist actions, observations, beliefs, smoothed beliefs, Viterbi path, and maze definition. Use the cell below in Colab to download everything as a ZIP.

In [ ]:
import os, json
os.makedirs("artifacts", exist_ok=True)

# Save arrays
np.savez("artifacts/run_data.npz",
         actions=np.array(actions, dtype='<U1'),
         observations=np.array(observations, dtype=int),
         true_states=np.array(true_states, dtype=int),
         beliefs=np.array(beliefs),
         smoothed=np.array(smoothed),
         viterbi=np.array(path, dtype=int),
         grid=maze.grid,
         colors=maze.colors)

# Save a few heatmaps as images (first, middle, last)
def save_heatmap_img(grid_b, fname, title):
    fig = plt.figure(figsize=(4,4))
    plt.imshow(grid_b, interpolation='nearest')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    fig.savefig(fname, dpi=120)
    plt.close(fig)

save_heatmap_img(belief_to_grid(beliefs[0], maze), "artifacts/belief_t0.png", "Belief t=0")
mid = len(beliefs)//2
save_heatmap_img(belief_to_grid(beliefs[mid], maze), f"artifacts/belief_t{mid}.png", f"Belief t={mid}")
save_heatmap_img(belief_to_grid(beliefs[-1], maze), "artifacts/belief_tT.png", "Belief t=T")

print("Saved artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 13. Extensions & Exercises

- **Alternative sensors**: wall-distance sensors in 4 directions; Gaussian noise.
- **Action noise**: add orientation states, differential-drive kinematics.
- **Partial map**: uncertain walls; treat map as latent to some degree.
- **Smoothing-based trajectories**: compare filtered, smoothed, and Viterbi positions quantitatively.
- **Scaling**: benchmark larger mazes; use sparse transitions and vectorized batch updates.